# 02. OAuth 2.0 Internals

## Background

OAuth 2.0 is the authorization framework underpinning virtually every modern web API, mobile app, and machine-to-machine service. OAuth 1.0 required cryptographic signatures on every request — correct but operationally painful. OAuth 2.0 replaced signatures with bearer tokens over TLS, which is simpler but shifts all security to transport and token management.

## What You'll Learn

- Authorization Code + PKCE flow: preventing code interception attacks
- Client credentials flow: machine-to-machine authentication
- Token introspection and scope intersection
- Common OAuth misconfigurations: open redirectors, missing state, implicit flow leakage

## Why This Matters

Every LLM agent calling external APIs uses OAuth. The 2022 GitHub OAuth breach and countless redirect-based attacks exploited gaps in OAuth implementations. Understanding the wire-level flow makes both implementation and vulnerability hunting tractable.

In [ ]:
import hashlib, hmac, secrets, time, json, base64
from dataclasses import dataclass, field
from typing import Optional

def b64url_encode(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).rstrip(b'=').decode()

def b64url_decode(s: str) -> bytes:
    pad = 4 - len(s) % 4
    if pad != 4: s += '=' * pad
    return base64.urlsafe_b64decode(s)


## 1. PKCE (Proof Key for Code Exchange)

PKCE (RFC 7636) prevents an attacker who intercepts the authorization code from exchanging it. The client generates a secret `code_verifier`, sends its SHA-256 hash as `code_challenge` during authorization, then proves knowledge of the verifier at token exchange.

In [ ]:
class PKCEChallenge:
    def __init__(self):
        self.code_verifier = secrets.token_urlsafe(64)
        raw = hashlib.sha256(self.code_verifier.encode()).digest()
        self.code_challenge = b64url_encode(raw)

pkce = PKCEChallenge()
print(f'code_verifier  (secret, client-side): {pkce.code_verifier[:40]}...')
print(f'code_challenge (sent to auth server): {pkce.code_challenge[:40]}...')

# PKCE verify
def pkce_verify(verifier: str, challenge: str) -> bool:
    raw = hashlib.sha256(verifier.encode()).digest()
    return hmac.compare_digest(b64url_encode(raw), challenge)

print(f'Verify correct verifier: {pkce_verify(pkce.code_verifier, pkce.code_challenge)}')
print(f'Verify wrong verifier:   {pkce_verify(secrets.token_urlsafe(64), pkce.code_challenge)}')


## 2. Authorization Code Flow Implementation

In [ ]:
@dataclass
class AuthorizationServer:
    client_id:     str
    redirect_uris: list = field(default_factory=list)
    _codes:  dict = field(default_factory=dict, repr=False)
    _tokens: dict = field(default_factory=dict, repr=False)

    def authorize(self, client_id, redirect_uri, scope, state, code_challenge):
        if client_id != self.client_id:
            return None, 'invalid_client'
        if redirect_uri not in self.redirect_uris:
            return None, 'invalid_redirect_uri'
        if not state:
            return None, 'state_required'
        code = secrets.token_urlsafe(32)
        self._codes[code] = {
            'client_id': client_id, 'redirect_uri': redirect_uri,
            'scope': scope, 'code_challenge': code_challenge,
            'expires_at': time.time() + 60, 'user_id': 'user_123',
        }
        return code, None

    def token_exchange(self, code, client_id, redirect_uri, code_verifier):
        entry = self._codes.pop(code, None)
        if entry is None: return None, 'invalid_grant'
        if time.time() > entry['expires_at']: return None, 'code_expired'
        if entry['client_id'] != client_id or entry['redirect_uri'] != redirect_uri:
            return None, 'invalid_grant'
        raw = hashlib.sha256(code_verifier.encode()).digest()
        challenge = b64url_encode(raw)
        if not hmac.compare_digest(challenge, entry['code_challenge']):
            return None, 'pkce_failed'
        access_token  = secrets.token_urlsafe(48)
        refresh_token = secrets.token_urlsafe(48)
        self._tokens[access_token] = {
            'user_id': entry['user_id'], 'scope': entry['scope'],
            'expires_at': time.time() + 3600,
        }
        return {'access_token': access_token, 'refresh_token': refresh_token,
                'token_type': 'Bearer', 'expires_in': 3600}, None


server = AuthorizationServer('myapp', ['https://myapp.example.com/callback'])
pkce2  = PKCEChallenge()
state  = secrets.token_urlsafe(16)
code, err = server.authorize('myapp', 'https://myapp.example.com/callback',
                              'read:data', state, pkce2.code_challenge)
print(f'Auth code: {code[:25]}...')
tokens, err = server.token_exchange(code, 'myapp',
                                    'https://myapp.example.com/callback', pkce2.code_verifier)
print(f'Access token: {tokens["access_token"][:25]}...')

# Attacker intercepts code but lacks verifier
code3, _ = server.authorize('myapp', 'https://myapp.example.com/callback',
                             'read:data', secrets.token_urlsafe(16), pkce2.code_challenge)
_, err_attack = server.token_exchange(code3, 'myapp',
                                      'https://myapp.example.com/callback',
                                      secrets.token_urlsafe(64))  # wrong verifier
print(f'Attacker with intercepted code: error={err_attack}')


## 3. Client Credentials (M2M)

In [ ]:
CLIENTS = {
    'svc_reports':   ('reports_secret',   ['read:metrics','read:data']),
    'svc_ingest':    ('ingest_secret',    ['write:data']),
    'svc_llm_agent': ('agent_secret',     ['llm_tool_call','read:data']),
}

def client_credentials_flow(client_id, client_secret, scope):
    if client_id not in CLIENTS: return None, 'invalid_client'
    stored_secret, allowed = CLIENTS[client_id]
    if not hmac.compare_digest(stored_secret, client_secret): return None, 'invalid_client'
    granted = set(scope.split()) & set(allowed)
    if not granted: return None, 'insufficient_scope'
    token = secrets.token_urlsafe(48)
    return {'access_token': token, 'scope': ' '.join(sorted(granted))}, None

for cid, sec, scope in [
    ('svc_reports','reports_secret','read:metrics read:data'),
    ('svc_llm_agent','agent_secret','llm_tool_call read:data write:data'),
    ('svc_ingest','WRONG','write:data'),
]:
    tok, err = client_credentials_flow(cid, sec, scope)
    if tok: print(f'{cid}: granted scopes={tok["scope"]}')
    else:   print(f'{cid}: ERROR={err}')


## 4. Common OAuth Misconfigurations

In [ ]:
misconfigs = [
    ('Open Redirector',      'wildcard redirect_uri',      'steal code via evil.example.com',  'exact URI match only'),
    ('Missing state param',  'no CSRF token in auth req',  'CSRF: attacker binds their token', 'cryptographic random state'),
    ('Implicit flow',        'token in URL fragment',       'exposed in logs/Referer headers',  'auth code + PKCE always'),
    ('Secret in mobile app', 'client_secret in APK',        'decompile to get secret',          'public client + PKCE'),
    ('Scope over-granting',  'no scope intersection check', 'request admin get admin',          'intersect with registered scopes'),
]
print(f'{"Misconfiguration":<25} {"Attack":<45} Fix')
print('-' * 110)
for name, vuln, attack, fix in misconfigs:
    print(f'{name:<25} {attack:<45} {fix}')
